<a href="https://colab.research.google.com/github/rahulpanigrahy650-droid/deep_learning_1/blob/main/cnn_image_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10


In [ ]:
#Datasets & dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),# convert to tensor and scale it
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))

])

trainset = CIFAR10(root="./data",train= True,download =True,transform=transform)
testset = CIFAR10(root="./data",train=False,download=True,transform=transform)


100%|██████████| 170M/170M [28:42<00:00, 99.0kB/s]


In [ ]:
trainloader = DataLoader(trainset,batch_size= 64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

Build the CNN

In [ ]:
from torch.nn.modules.linear import Linear
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3,32,kernel_size =3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2), # kernel size =2,stride =2

        nn.Conv2d(32,64,kernel_size =3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2), # kernel size =2,stride =2

        nn.Conv2d(64,128,kernel_size =3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2) # kernel size =2,stride =2

)
    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),
        nn.Linear(256,10)
    )

  def forward(self,x):
    x = self.conv_layers(x)
    x = x.view(x.size(0), -1) # flattening
    x = self.fc_layers(x)

    return x

In [ ]:
model = CNN()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

Trainning ANN

In [ ]:
epochs = 10
train_loss = []
val_loss = []
for epoch in range(epochs):
  epoch_trainning_loss =0.0

  for image,labels in trainloader:
    optimizer.zero_grad()

    output = model.forward(image) #fp
    loss = criterion(output,labels) # loss fnx
    loss.backward() #bp
    optimizer.step() # update params

    epoch_trainning_loss += loss.item()
train_loss.append(epoch_trainning_loss/len(trainloader))
model.eval()
epoch_val_loss =0.0

with torch.no_grad():
  for image,labels in testloader:
    output = model.forward(image)
    loss = criterion(output,labels)
    epoch_val_loss += loss.item()

val_loss.append(epoch_val_loss/len(testloader))

print(f"epoch={epoch+1}/{epochs} ==> train loss = {epoch_trainning_loss/len(trainloader)} & val loss ={epoch_val_loss/len(testloader)} ")

In [ ]:
import matplotlib.pyplot as plt
loss_df = pd.DataFrame({
    "trainning_loss":train_loss,
    "val_loss" : val_loss
})

plt.figure(figsize=(10,5))
plt.plot(loss_df["trainning_loss"],label = "trainning_loss")
plt.plot(loss_df["val_loss"],label ="val_loss")

plt.title("Trainning loss & Validation loss")
plt.xlabel("epochs")
plt.ylabel("loss")

plt.legend()
plt.show()

NameError: name 'pd' is not defined

In [ ]:
#Evaluation our cnn

correct_labels = 0
total_labels =0

model.eval()

with torch.no_grad():
  for image,labels in testloader:
    outputs= model.forward(image)
    _,predicted = torch.max(outputs,1)

    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

print(f"accuracy = {correct_labels/total_labels *100}")